# Parking Space Occupancy Detection

**CAI2840C — Research experiment**  
Compare **baseline CNN**, **MobileNetV3**, **VGG16**, and **ResNet50** on PKLot + CNRPark-EXT (or demo data).

Run from the `Parking` project root (kernel working directory = project root).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "run_pipeline.py").exists():
    pass
elif (ROOT.parent / "run_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## 1. Setup & config

In [ ]:
from src.utils import load_config, ensure_dirs, set_seed

cfg = load_config()
# Faster smoke run in the notebook; raise for the full study
cfg["epochs"] = 5
set_seed(cfg["seed"])
paths = ensure_dirs(cfg)
cfg

## 2. Data inventory

Use `--demo` synthetic patches to verify the pipeline, or place real PKLot / CNRPark-EXT under `data/raw/`.

In [ ]:
from src.download_datasets import create_demo_dataset, check_raw
from src.prepare_data import prepare

USE_DEMO = True  # set False when real datasets are in data/raw/

if USE_DEMO:
    create_demo_dataset(n_per_class=60)
else:
    check_raw()

prepare()
import pandas as pd
df = pd.read_csv(paths["processed"] / "split_manifest.csv")
df.groupby(["split", "dataset", "label_name"]).size().unstack(fill_value=0)

## 3. Train models

In [ ]:
from src.train import train_one

MODELS = ["baseline", "mobilenetv3"]  # add "vgg16", "resnet50" for full comparison
train_metas = {}
for name in MODELS:
    print("=" * 60, name)
    train_metas[name] = train_one(name, cfg)

## 4. Evaluate on held-out test set

In [ ]:
from src.evaluate import evaluate_model, summarize_all
from IPython.display import Image, display

for name in MODELS:
    evaluate_model(name, cfg)
    fig = paths["figures"] / f"{name}_confusion_matrix.png"
    if fig.exists():
        display(Image(filename=str(fig)))

summarize_all(cfg)

## 5. Failure-case review (vision-only limitations)

Inspect false positives / false negatives by weather and dataset for H3.

In [ ]:
from pathlib import Path

for name in MODELS:
    fail_path = paths["metrics"] / f"{name}_failures.csv"
    if fail_path.exists():
        fails = pd.read_csv(fail_path)
        print(f"\n{name}: {len(fails)} errors")
        if len(fails):
            display(fails.groupby(["weather", "label_name"]).size().unstack(fill_value=0))
            display(fails.head(10))